## 2. Data Cleaning ##

## 2.a Import dan panggil data mentah sebelumnya ##

In [1]:
import polars as pl
import duckdb

RAW_PATH = "/workspace/data/raw/fire_archive_SV-C2_812847.csv"
lf = pl.scan_csv(RAW_PATH)
n_raw = lf.select(pl.len()).collect().item()
print(f"Baris mentah: {n_raw:,}")


Baris mentah: 1,535,566


## PENJELASAN ##
1. Data dibaca dengan scan_csv(lazy), yang akan menjadikan file tidak dimuat penuh ke memori. Polars akan memprosesnya saat method .collect() dipanggil. Hasil pemanggilan jumlah data mentah dengan pemanggilan ini sama yaitu 1.535.566 baris.

## 2.b Validasi Duplikasi ##

In [2]:
dup_check = (
    lf.group_by(["latitude", "longitude", "acq_date", "acq_time"])
    .agg(pl.len().alias("jumlah"))
    .filter(pl.col("jumlah") > 1)
    .collect()
)
print(f"Baris dengan kombinasi kunci duplikat: {dup_check.height:}")
dup_check.head(5)

Baris dengan kombinasi kunci duplikat: 0


latitude,longitude,acq_date,acq_time,jumlah
f64,f64,str,i64,u32


## PENJELASAN ##
1. Dalam cell ini, satu titik hotspot dianggap unik berdasarkan kombinasi latitude, longitude, tanggal, dan jam akuisisi. Data dikelompokkkan berdasarkan kombinasi itu, lalu dicari yang muncul lebih dari satu kali. Hasilnya 0 baris, jadi tidak ada duplikasi dan tidak ada yang perlu dihapus.

## 2.c Filtering ##

In [3]:
FILTER_EXCLUDE_LOW_CONFIDENCE = True
FILTER_ONLY_VEGETATION_FIRES = True

lf_clean = lf
if FILTER_EXCLUDE_LOW_CONFIDENCE:
    lf_clean = lf_clean.filter(pl.col("confidence") != "l")
if FILTER_ONLY_VEGETATION_FIRES:
    lf_clean = lf_clean.filter(pl.col("type") == 0)

n_clean =lf_clean.select(pl.len()).collect().item()
print(f"Baris sebelum filter : {n_raw:,}")
print(f"Baris setelah filter : {n_clean:,}")
print(f"Persentase dibuang : {(1 - n_clean / n_raw) * 100:.2f}%")


Baris sebelum filter : 1,535,566
Baris setelah filter : 1,317,362
Persentase dibuang : 14.21%


## PENJELASAN ##
1. Cell ini fokus pada 2 filter, yaitu bagian confidence != 'l', dimana baris ini mendefinisikan mengenai piksel siang hari yang berpotensi terkena pantulan matahari dengan sinyal suhu lemah, yang membuatnya lebih rawan terhadap false positive.
2. Bagian lainnya adalah type == 0, dimana kolom ini mendefinisikan tipe titik api seperti gunung berapi, sumber panas dari industri dan offshore (laut), bukan fokus pada kebakaran hutan/lahan, sehingga akan membuat angkanya melambung besar jika dibiarkan.
3. Hasilnya = dari 1.535.566 baris, tersisa 1,317,362 (14.21% dibuang). Kenapa bisa terjadi? karena kolom low confidence sekitar 6,6% dan non-vegetation sekitar 7,8%, dan sebagian baris masuk di keduanya, membuat totalnya sedikit di bawah penjumlahan keduanya.

## Cek Outliers ##

In [5]:
num_cols = ["brightness", "bright_t31", "frp", "scan", "track"]

stats = lf_clean.select(
    [pl.col(c).min().alias(f"{c}_min") for c in num_cols] +
    [pl.col(c).quantile(0.99).alias(f"{c}_p99") for c in num_cols] +
    [pl.col(c).max().alias(f"{c}_max") for c in num_cols]
).collect()
print(stats.transpose(include_header=True, column_names=["nilai"]))

for c in num_cols:
    q1, q3 = lf_clean.select(
        pl.col(c).quantile(0.25).alias("q1"), pl.col(c).quantile(0.75).alias("q3")
    ).collect().row(0)
    iqr = q3 - q1
    batas_atas = q3 + 1.5 * iqr
    n_out = lf_clean.filter(pl.col(c) > batas_atas).select(pl.len()).collect().item()
    print(f"{c}: batas atas IQR = {batas_atas:.2f}, outlier = {n_out:,}")

shape: (15, 2)
┌────────────────┬─────────┐
│ column         ┆ nilai   │
│ ---            ┆ ---     │
│ str            ┆ f64     │
╞════════════════╪═════════╡
│ brightness_min ┆ 295.0   │
│ bright_t31_min ┆ 150.0   │
│ frp_min        ┆ 0.0     │
│ scan_min       ┆ 0.32    │
│ track_min      ┆ 0.36    │
│ …              ┆ …       │
│ brightness_max ┆ 367.0   │
│ bright_t31_max ┆ 378.18  │
│ frp_max        ┆ 1322.99 │
│ scan_max       ┆ 0.8     │
│ track_max      ┆ 0.78    │
└────────────────┴─────────┘
brightness: batas atas IQR = 362.85, outlier = 45,471
bright_t31: batas atas IQR = 313.19, outlier = 7,083
frp: batas atas IQR = 17.17, outlier = 113,671
scan: batas atas IQR = 0.67, outlier = 30,070
track: batas atas IQR = 0.81, outlier = 0


## PENJELASAN ##
1. Dari hasil cell ini, didapat kesimpulan berikut:
- frp dengan outlier sebanyak 113,671, dimana DISTRIBUSI FRP ini selalu miring ke kanan karena dominan titik api kecil, sedikit untuk titik api besar. Karena IQR mengasumsikan distribusi yang kira kira simetris, maka hasilnya memnadai banyak baris yang sebenarnya normal. Selain itu, jika terdapat nilai di atas batas itu, bukan data rusak melainkan memang terdapat titik api besar. Jika membuang data tersebut akan menghilangkan juga kejadian kejadian real-time kebakaran hutan.
- brightness dengan outlier sebanyak 45,471, dimana brightness ini merupakan angka untuk suhu kecerahan kanal api dalam satuan Kelvin.
- bright_t31 ini menggambarkan suhu latar belakang piksel, bukan suhu api.
- scan menjelaskan mengenai ukuran piksel VIIRS yang membesar ke arah tepi lebar sapuan satelit (Timur-Barat)
- track ini berhubungan dengan nilai scan, dimana ukuran pikselnya membesar pada arah Utara-Selatan

## 2.d Spatial Join ( Titik Hotspot Provinsi) ##


In [5]:
import sys
!{sys.executable} -m pip install geopandas

In [6]:
import geopandas as gpd
print(gpd.__version__)

1.2.0


In [7]:
import geopandas as gpd

BOUNDARY_PATH = "/workspace/data/external/indonesia_provinces.geojson"

provinces = gpd.read_file(BOUNDARY_PATH)
print(provinces.columns.tolist())
provinces.head()

['GID_1', 'GID_0', 'COUNTRY', 'NAME_1', 'VARNAME_1', 'NL_NAME_1', 'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'geometry']


,GID_1,GID_0,COUNTRY,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,IDN.1_1,IDN,Indonesia,Aceh,NA,NA,Propinisi,Province,11,ID.AC,ID-AC,"MULTIPOLYGON (((96.6813 2.0974, 96.6872 2.0905..."
1,IDN.2_1,IDN,Indonesia,Bali,NA,NA,Propinisi,Province,51,ID.BA,ID-BA,"MULTIPOLYGON (((115.5257 -8.8045, 115.5254 -8...."
2,IDN.3_1,IDN,Indonesia,BangkaBelitung,NA,NA,Propinisi,Province,19,ID.BB,NA,"MULTIPOLYGON (((108.0137 -3.2445, 108.0108 -3...."
3,IDN.4_1,IDN,Indonesia,Banten,NA,NA,Propinisi,Province,36,ID.BT,ID-BT,"MULTIPOLYGON (((106.3868 -6.9907, 106.385 -6.9..."
4,IDN.5_1,IDN,Indonesia,Bengkulu,NA,NA,Propinisi,Province,17,ID.BE,ID-BE,"MULTIPOLYGON (((103.573 -4.923, 103.5713 -4.92..."


In [13]:
PROVINCE_NAME_COLUMN = "NAME_1"

df_clean = lf_clean.select(
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "confidence",
    "type"
).collect().to_pandas()

points = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean["longitude"], df_clean["latitude"]),
    crs="EPSG:4326",
)

provinces = provinces.to_crs("EPSG:4326")

joined = gpd.sjoin(points, provinces[[PROVINCE_NAME_COLUMN, "geometry"]], how="left", predicate="within")

n_unmatched = joined[PROVINCE_NAME_COLUMN].isna().sum()
print(f"Titik yang tidak ter-assign ke provinsi manapun: {n_unmatched:,} ({n_unmatched / len(joined) * 100:.2f}%)")

print(f"Jumlah baris sebelum drop unmatched: {len(joined):,}")
joined = joined[joined[PROVINCE_NAME_COLUMN].notna()].copy()
print(f"Jumlah baris setelah drop unmatched: {len(joined):,}")

print("Kemungkinan penyebab: titik di laut (offshore) atau titik yang persis di garis batas")



Titik yang tidak ter-assign ke provinsi manapun: 2,880 (0.22%)
Jumlah baris sebelum drop unmatched: 1,317,364
Jumlah baris setelah drop unmatched: 1,314,484
Kemungkinan penyebab: titik di laut (offshore) atau titik yang persis di garis batas


## PENJELASAN ##
1. Setiap titik hotspot diubah menjadi objek geometri (titik) dari longitude dan latitude, alu dicocokkan dengan polygon provinsi dari sumber https://gadm.org/download_country.html dengan predikat within. kedua data memakai CRS yang sama agar posisinya cocok. Titik yang tidak jatuh di polygon provinsi mana pun mendapat nilai null dengan jumlah 2,880 titik (0.22% dari data).
2. Titik tanpa provinsi dibuang dengan sebab titik besar kemungkinan titik berada di laut atau tepat di garis batas. Unit analisis kita adalah provinsi daratan, dan memaksa titik-titik ini masuk ke provinsi terdekat akan menambah asumsi yang sulit dibenarkan. Dengan baris sebelum drop unmatched: 1,317,364 baris menjadi setelah drop unmatched sebanyak 1,314,484 baris.


## 2.e Agregasi Unit Analisis: Provinsi x Bulan, dinormalisasi per luas wilayah ##

In [14]:
joined_pl = pl.from_pandas(joined.drop(columns="geometry"))

agg = (
    joined_pl
    .with_columns(
        pl.col("acq_date").str.to_date().dt.strftime("%Y-%m").alias("tahun_bulan")
    )
    .group_by([PROVINCE_NAME_COLUMN, "tahun_bulan"])
    .agg(pl.len().alias("jumlah_deteksi"))
    .sort([PROVINCE_NAME_COLUMN, "tahun_bulan"])
)
agg.head(10)

NAME_1,tahun_bulan,jumlah_deteksi
str,str,u32
"""Aceh""","""2016-01""",205
"""Aceh""","""2016-02""",442
"""Aceh""","""2016-03""",849
"""Aceh""","""2016-04""",502
"""Aceh""","""2016-05""",119
"""Aceh""","""2016-06""",429
"""Aceh""","""2016-07""",571
"""Aceh""","""2016-08""",503
"""Aceh""","""2016-09""",410


## PENJELASAN ##
1. Cell ini menggabungkan nama provinsi dengan kolom tahun/bulan serta jumlah deteksi tiap bulannya.

In [15]:
provinces_equal_area = provinces.to_crs("EPSG:6933")

luas_provinsi = (
    provinces_equal_area
    .assign(luas_km2=provinces_equal_area.geometry.area / 1_000_000)  # m2 -> km2
    [[PROVINCE_NAME_COLUMN, "luas_km2"]]
)

print(luas_provinsi.sort_values("luas_km2", ascending=False).head(10))

import os
os.makedirs("/workspace/data/external", exist_ok=True)
luas_provinsi.to_csv("/workspace/data/external/luas_provinsi.csv", index=False)
print("Tersimpan: /workspace/data/external/luas_provinsi.csv")

              NAME_1       luas_km2
22             Papua  313773.326492
13  KalimantanTengah  153711.086199
11   KalimantanBarat  146869.654239
14   KalimantanTimur  125829.729561
23        PapuaBarat   98500.307117
24              Riau   89874.498184
31   SumateraSelatan   86986.344561
32     SumateraUtara   72229.064051
15   KalimantanUtara   70114.152441
27    SulawesiTengah   61156.202508
Tersimpan: /workspace/data/external/luas_provinsi.csv


## PENJELASAN ##
1. Cell ini fokus untuk menghitung luas tiap provinsi menggunakan Polygon GADM lalu diubah ke proyeksi equal-area (EPSG:6933) lalu dikonversi dari m^2 ke km^2. Proses ini dibutuhkan karena luas di sistem lat/lon biasa dalam satu derajat persegi, bukan km^2. Hasilnya disimpan ke luas_provinsi.csv. Data atau nilai bisa berbeda dari angka BPS karena perbedaan presisi batas wilayah.

In [16]:
luas_provinsi = pl.read_csv("/workspace/data/external/luas_provinsi.csv")

agg_normalized = (
    agg.join(luas_provinsi, on=PROVINCE_NAME_COLUMN, how="left")
    .with_columns(
        (pl.col("jumlah_deteksi") / pl.col("luas_km2") * 1000).alias("deteksi_per_1000km2")
    )
)

# Cek provinsi yang gagal join (nama tidak cocok antara data FIRMS hasil spatial join dan tabel luas)
n_gagal_join = agg_normalized.filter(pl.col("luas_km2").is_null()).select(PROVINCE_NAME_COLUMN).unique()
print(f"Provinsi yang gagal dapat data luas wilayah: {n_gagal_join.height}")
if n_gagal_join.height > 0:
    print(n_gagal_join)

agg_normalized.head(10)

Provinsi yang gagal dapat data luas wilayah: 0


NAME_1,tahun_bulan,jumlah_deteksi,luas_km2,deteksi_per_1000km2
str,str,u32,f64,f64
"""Aceh""","""2016-01""",205,56836.400597,3.606843
"""Aceh""","""2016-02""",442,56836.400597,7.776706
"""Aceh""","""2016-03""",849,56836.400597,14.93761
"""Aceh""","""2016-04""",502,56836.400597,8.832368
"""Aceh""","""2016-05""",119,56836.400597,2.093729
"""Aceh""","""2016-06""",429,56836.400597,7.54798
"""Aceh""","""2016-07""",571,56836.400597,10.046379
"""Aceh""","""2016-08""",503,56836.400597,8.849962
"""Aceh""","""2016-09""",410,56836.400597,7.213687


## PENJELASAN ##
1. Cell ini fokus untuk mengelompokkkan data per provinsi dan per bulan, lalu dihitung jumlah deteksinya. Satu baris di tabel hasil mewakili satu kombinasi provinsi-bulan.
2. Tabel agregasi digabung dengan tabel luas wilayah, lalu dihitung deteksi_per_1000km2 = jumlah deteksi / luas x 1000. Proses tersebut diperlukan agar provinsi besar tidak otomatis berlabel "lebih rawan" hanya karena wilayahnya luas. 

## 2.f Validasi hasil dengan DuckDB ##

In [17]:
import os
os.makedirs("/workspace/data/processed", exist_ok=True)
OUTPUT_PATH = "/workspace/data/processed/hotspot_provinsi_bulanan.parquet"
agg_normalized.write_parquet(OUTPUT_PATH)
print(f"Tersimpan: {OUTPUT_PATH}")

Tersimpan: /workspace/data/processed/hotspot_provinsi_bulanan.parquet


## PENJELASAN ##
1. Hasil akhir disimpan dalam parquet yang memiliki format kolumnar dan terkompresi, sehingga file menjadi kecil dan cepat dibaca ulang.

In [18]:
con = duckdb.connect()
total_duckdb = con.execute(f"SELECT SUM(jumlah_deteksi) FROM '{OUTPUT_PATH}'").fetchone()[0]
total_polars = agg_normalized["jumlah_deteksi"].sum()

print(f"Total DuckDB : {total_duckdb:,}")
print(f"Total Polars : {total_polars:,}")
assert total_duckdb == total_polars, f"Total deteksi tidak sama: DuckDB={total_duckdb}, Polars={total_polars}"
print("Validasi lolos: total deteksi di file parquet konsisten dengan hasil agregasi Polars.")

Total DuckDB : 1,314,484
Total Polars : 1,314,484
Validasi lolos: total deteksi di file parquet konsisten dengan hasil agregasi Polars.


## PENJELASAN ##
1. Total deteksi dihitung ulang dari file Parquet memakai DuckDB lalu dibandingkan dengan total di Polars. Jika keduanya sama maka proses penyimpanan dan pembacaan ulang tidak mengubah data.

In [ ]:
import os
for root, dirs, files in os.walk("../data"):
    for f in files:
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        print(f"{path}  ({size_kb:.1f} KB)")

../data/processed/hotspot_provinsi_bulanan.parquet  (29.8 KB)
../data/external/luas_provinsi.csv  (1.0 KB)


## PENJELASAN ##
1. Cell ini dibuat karena sebelumnya tidak terdetect dimana file parquet dan file csv yang sudah dibuat. Codingan ini membantu untuk mencari tau tempat file file tersebut.